# SCADA 라벨 정제 실험

미사용 자산인 원본 SCADA(VESTAS 12기·UNISON 5기, 10분)를 사용해 **"바람 있는데 발전 0"(고장·출력제한) 시간을 식별**,
학습 라벨에서 제외/가중축소가 blend 예측을 개선하는지 2폴드로 판정.

- 그룹 매핑(info.xlsx): VESTAS 1-6 → g1, VESTAS 7-12 → g2, UNISON 1-5 → g3
- stuck 터빈-시간: 풍속≥4 m/s인데 출력≤정격 1% (VESTAS 3.6MW, UNISON 4.2MW)
- 그룹-시간 마스크: 보고 터빈≥3 중 stuck 비율≥30%
- 변형: W1.0(기준선) / W0.5 / W0.2 / W0.0(제외) — 학습 가중치만 변경, 검증은 전체
- **주의**: 라벨(실발전)에는 미래에도 정지가 섞이므로 정제가 항상 이득은 아님 — 그래서 실험으로 판정
- SCADA는 학습 가중치 산출에만 사용(테스트 입력 아님 → 규칙 위반 없음, CONSTRAINTS §4)

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

# ── SCADA → 그룹-시간 stuck 비율 ──
def stuck_table():
    out={}
    specs=[("scada_vestas_train.csv","vestas",12,3600.0),("scada_unison_train.csv","unison",5,4200.0)]
    frames={}
    for fn,pre,n,rate in specs:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"), ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1)
        rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        fr=(st/rp).where(rp>=3)          # 보고 터빈 3개 미만이면 판단 보류(NaN→마스크 안함)
        return fr
    out[1]=frac("vestas",range(1,7)); out[2]=frac("vestas",range(7,13)); out[3]=frac("unison",range(1,6))
    return out
STUCK=stuck_table()
for g in GROUPS:
    fr=FR[g]; s=STUCK[g].reindex(fr.kst_dtm).to_numpy()
    masked=(np.nan_to_num(s,nan=0.0)>=0.3)
    FR[g]=fr.assign(stuck_mask=masked)
    print(f"g{g}: stuck 마스크 시간 {masked.sum()} ({100*masked.mean():.1f}%)")

g1: stuck 마스크 시간 4727 (18.0%)
g2: stuck 마스크 시간 4872 (18.6%)
g3: stuck 마스크 시간 778 (4.4%)


In [2]:
class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)

def train_one(pool_tr,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    net=MLP(len(FEATS),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def blend_predict(tr_frames, stuck_w):
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[FEATS+["kst_dtm","stuck_mask"]].copy()
        p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
        p["w"]=np.where(p["stuck_mask"],stuck_w,1.0); pool.append(p)
    pool=pd.concat(pool,ignore_index=True)
    acc={g:[] for g in tr_frames}
    for sd_ in SEEDS:
        net,scaler=train_one(pool,sd_)
        for g,(_,va2) in tr_frames.items():
            acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]; tgt=TGT[g]
        w=np.where(tr2["stuck_mask"],stuck_w,1.0)
        lg_=lgbm().fit(tr2[FEATS],tr2[tgt],sample_weight=w)
        hg_=histgbm().fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
        gp=np.clip(0.5*(lg_.predict(va2[FEATS])+hg_.predict(va2[FEATS].to_numpy())),0,cap)
        out[g]=np.clip((1-BLEND)*gp+BLEND*np.mean(acc[g],axis=0),0,cap)
    return out

FOLDS={2023:[2022],2024:[2022,2023]}
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent

def fold_total(preds,vy):
    nm=[]; fi=[]
    for g,p in preds.items():
        tgt=TGT[g]; yt=CACHE[vy][g][1][tgt].to_numpy()
        a,b=group_scores(yt,p,W.CAP[g]); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)

res={}
for wv in [1.0,0.5,0.2,0.0]:
    tot={}
    for vy,ent in CACHE.items():
        preds=blend_predict({g:e for g,e in ent.items()},stuck_w=wv)
        tot[vy]=fold_total(preds,vy)
    res[wv]=tot
    print(f"stuck_w={wv}: 2023={tot[2023]:.4f}  2024={tot[2024]:.4f}  min={min(tot.values()):.4f}")

base=res[1.0]
cands=[w for w in [0.5,0.2,0.0] if res[w][2023]>=base[2023] and res[w][2024]>=base[2024]]
winner=max(cands,key=lambda w:min(res[w].values())) if cands else 1.0
print(f"\n판정: stuck_w={winner}" + (" 채택" if winner!=1.0 else " (개선 없음 → 정제 미적용)"))
json.dump(dict(totals={str(w):{str(k):round(v,4) for k,v in res[w].items()} for w in res}, winner=winner),
          open("submission/ver_7/scada_clean_result.json","w"),ensure_ascii=False,indent=2)
print("saved scada_clean_result.json")

stuck_w=1.0: 2023=0.5986  2024=0.6084  min=0.5986


stuck_w=0.5: 2023=0.6098  2024=0.6162  min=0.6098


stuck_w=0.2: 2023=0.6070  2024=0.6176  min=0.6070


stuck_w=0.0: 2023=0.6093  2024=0.6202  min=0.6093

판정: stuck_w=0.5 채택
saved scada_clean_result.json
